# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells (FAIR^2) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, and explored using identifiers (`@id`) for all record sets and fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant Schema JSON-LD)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Explore available record sets, fields, and their `@id`. 

**Referencing by `@id`:**
- Each record set, field, and column are referenced by their `@id` as per the Croissant schema.
- We'll enumerate record sets and their fields to determine what can be loaded.

In [ ]:
# List record sets and their fields by `@id`.
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecord Set Name: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
    # List fields in each record set
    field_names = []
    field_ids = []
    for f in rs.fields:
        field_names.append(getattr(f, 'name', 'N/A'))
        field_ids.append(f.id)
    print(f"  Fields ({len(field_ids)}):")
    for n, fid in zip(field_names, field_ids):
        print(f"    - {n} (@id: {fid})")

## 3. Data Extraction
Load data from a **specific record set** into a DataFrame for analysis.

**Instructions:**
- Choose a record set (by `@id`) to extract records.
- You'll see examples using fields and columns all referenced by their `@id`.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for RecordSet: {rsid}")

# Show available columns for the first record set (as example)
main_record_set_id = record_set_ids[0]
print(f"\nFields (columns) for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Steps:**
- Select a numeric field (by `@id`) for filtering/normalizing.
- Filter values, normalize them, and optionally group by another field attribute.

In [ ]:
# Select the main record set and fields to analyze
record_set_id = main_record_set_id  # reuse, or set explicitly

# For this dataset, let's inspect typical field names
df = dataframes[record_set_id]
print("Available columns:", df.columns.tolist())
# For example, let's analyze protein abundance (if present)
# We'll look for likely candidates: e.g., fields with 'abundance', 'MW', 'count', 'peptide', etc.

numeric_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['abundance', 'mw', 'count', 'peptide', 'coverage', 'weight'])]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]

print(f"Chosen numeric field for analysis: {numeric_field}")

# Filter: Only keep rows where the numeric field > a threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is not None:
    filtered_df = df[df[numeric_field] > threshold]
else:
    # Try convert to numeric (if field is read as string)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field
category_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['group', 'category', 'type', 'class', 'sample']) and col != numeric_field]
if category_field_candidates:
    group_field = category_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the data distribution for the selected numeric field, and relationship to a grouping variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the chosen numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if category_field_candidates:
    # Boxplot by group
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- Explored the FAIR^2 proteomics dataset via its Croissant schema with `mlcroissant`.
- Enumerated available record sets, and loaded fields using their `@id`.
- Extracted records, performed basic filtering/normalization/grouping, and visualized numeric data.
- See the [FAIR^2 documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for further domain-specific context.